# 04 — Feature Selection, SHAP, and Final Model Recommendation

Loads `train.pkl`/`test.pkl` and `best_estimators.pkl` from notebook 03.
Investigates redundancy (correlation), mutual information, permutation
importance, and SHAP for the leading Basic and Enhanced candidates, then
uses that evidence to trim the Enhanced feature set and produces the final
recommended-model metrics.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, brier_score_loss)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
import shap

RANDOM_STATE = 42
TARGET = 'Heart Disease'
pd.set_option('display.width', 160)

train_df = pd.read_pickle("train.pkl")
test_df = pd.read_pickle("test.pkl")
with open("best_estimators.pkl", "rb") as f:
    best_estimators = pickle.load(f)

BASIC_NUMERIC = ['Age', 'Height (cm)', 'Weight (kg)', 'BP(mmHg)']
BASIC_BINARY = ['Family H/O', 'Hypertension', 'Diabetes', 'H/O ChestPain']
BASIC_CATEGORICAL = ['Sex']
ENHANCED_NUMERIC_ADD = ['Total_Cholesterol(mg/dL)', 'HDL(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'RBS(mmol/L)', 'MaxHR']

## 1. Mutual information (Basic + Enhanced numeric/binary features)

Univariate, model-free signal strength - a first-pass filter, not a final
decision (redundant features can each individually show high MI while
being interchangeable, which is exactly the Age/MaxHR situation checked
next).

In [2]:
enhanced_cols = BASIC_NUMERIC + ENHANCED_NUMERIC_ADD
mi_df = train_df[enhanced_cols + BASIC_BINARY].fillna(train_df[enhanced_cols + BASIC_BINARY].median(numeric_only=True))
mi_scores = mutual_info_classif(mi_df, train_df[TARGET], random_state=RANDOM_STATE)
pd.Series(mi_scores, index=mi_df.columns).sort_values(ascending=False).round(4)

LDL(mg/dL)                  0.3229
Total_Cholesterol(mg/dL)    0.2599
Triglycerides(mg/dL)        0.2237
RBS(mmol/L)                 0.1798
MaxHR                       0.1453
Age                         0.1322
H/O ChestPain               0.1145
BP(mmHg)                    0.0929
Hypertension                0.0559
Diabetes                    0.0554
Weight (kg)                 0.0519
Family H/O                  0.0441
Height (cm)                 0.0393
HDL(mg/dL)                  0.0135
dtype: float64

## 2. Redundancy / correlation matrix among Basic+Enhanced numeric features

In [3]:
corr = train_df[enhanced_cols].corr().round(2)
high_corr = [(c1, c2, corr.loc[c1, c2]) for i, c1 in enumerate(corr.columns) for c2 in corr.columns[i+1:] if abs(corr.loc[c1, c2]) >= 0.6]
print("Pairs with |r| >= 0.6:", high_corr)
corr

Pairs with |r| >= 0.6: [('Age', 'MaxHR', np.float64(-0.92)), ('Total_Cholesterol(mg/dL)', 'LDL(mg/dL)', np.float64(0.62))]


,Age,Height (cm),Weight (kg),BP(mmHg),Total_Cholesterol(mg/dL),HDL(mg/dL),LDL(mg/dL),Triglycerides(mg/dL),RBS(mmol/L),MaxHR
Age,1.00,-0.00,0.05,-0.02,0.32,-0.19,0.42,0.28,0.21,-0.92
Height (cm),-0.00,1.00,0.36,-0.12,0.04,-0.14,0.11,-0.02,0.07,0.20
Weight (kg),0.05,0.36,1.00,-0.00,0.36,-0.15,0.25,0.32,0.16,0.12
BP(mmHg),-0.02,-0.12,-0.00,1.00,-0.01,0.03,-0.01,0.04,-0.02,-0.02
Total_Cholesterol(mg/dL),0.32,0.04,0.36,-0.01,1.00,-0.19,0.62,0.46,0.28,-0.25
HDL(mg/dL),-0.19,-0.14,-0.15,0.03,-0.19,1.00,-0.29,-0.12,-0.14,0.07
LDL(mg/dL),0.42,0.11,0.25,-0.01,0.62,-0.29,1.00,0.46,0.37,-0.35
Triglycerides(mg/dL),0.28,-0.02,0.32,0.04,0.46,-0.12,0.46,1.00,0.29,-0.24
RBS(mmol/L),0.21,0.07,0.16,-0.02,0.28,-0.14,0.37,0.29,1.00,-0.18
MaxHR,-0.92,0.20,0.12,-0.02,-0.25,0.07,-0.35,-0.24,-0.18,1.00


**Age <-> MaxHR at r=-0.92** is the dominant redundancy (confirms the
notebook-01/notebook-02 finding from a third, independent angle).
**Total Cholesterol <-> LDL at r=0.62** is a secondary, expected redundancy
(Total Cholesterol is arithmetically related to LDL+HDL+Triglycerides/5) -
noted but not acted on below since both show strong individual importance.

## 3. Permutation importance (test set, ROC-AUC drop) for the leading Basic and Enhanced candidates (XGBoost)

In [4]:
for fs_name in ['A_Basic', 'B_Basic_Enhanced']:
    pipe = best_estimators[(fs_name, 'XGBoost')]
    result = permutation_importance(pipe, test_df, test_df[TARGET], n_repeats=30,
                                     random_state=RANDOM_STATE, scoring='roc_auc', n_jobs=-1)
    # feature_names_in_ reflects all columns of the DataFrame passed to .fit(), not just
    # the ones the ColumnTransformer actually selects - features outside this pipeline's
    # feature set correctly show ~0 importance below (a useful sanity check in itself).
    feature_names = list(pipe.named_steps['preprocess'].feature_names_in_)
    imp = pd.Series(result.importances_mean, index=feature_names).sort_values(ascending=False)
    used_cols = pipe.named_steps['preprocess'].transformers[0][2] + pipe.named_steps['preprocess'].transformers[1][2] + pipe.named_steps['preprocess'].transformers[2][2]
    print(f"=== {fs_name} / XGBoost - permutation importance, features actually used by this pipeline ===")
    print(imp.loc[imp.index.isin(used_cols)].round(4).to_string())
    print()

=== A_Basic / XGBoost - permutation importance, features actually used by this pipeline ===
Age              0.0915
BP(mmHg)         0.0581
H/O ChestPain    0.0482
Hypertension     0.0378
Weight (kg)      0.0219
Family H/O       0.0097
Height (cm)      0.0083
Diabetes         0.0069
Sex             -0.0000



=== B_Basic_Enhanced / XGBoost - permutation importance, features actually used by this pipeline ===
LDL(mg/dL)                  0.0576
RBS(mmol/L)                 0.0208
Triglycerides(mg/dL)        0.0202
Total_Cholesterol(mg/dL)    0.0171
BP(mmHg)                    0.0115
H/O ChestPain               0.0088
Hypertension                0.0068
Diabetes                    0.0046
Age                         0.0022
Weight (kg)                 0.0010
Family H/O                  0.0008
MaxHR                       0.0006
HDL(mg/dL)                  0.0002
Height (cm)                 0.0002
Sex                         0.0000



**`BP(mmHg)` and `H/O ChestPain` both show real, positive multivariate
importance** in the Basic model (0.058 and 0.048 AUC-drop respectively)
despite `BP`'s near-zero *univariate* correlation with the target noted in
`dataset_audit.md` - directly validating the instruction to keep them in
Basic and confirm usefulness via feature-selection experiments rather than
univariate correlation alone.

**In the Enhanced model, `MaxHR` (0.0006) and `HDL` (0.0002) show
negligible incremental importance** once `Age`, the rest of the lipid
panel, and `RBS` are present - tested for removal below.

## 4. SHAP global importance (test set) for the same two candidates

In [5]:
shap_results = {}
for fs_name in ['A_Basic', 'B_Basic_Enhanced']:
    pipe = best_estimators[(fs_name, 'XGBoost')]
    pre, model = pipe.named_steps['preprocess'], pipe.named_steps['model']
    X_test_transformed = pre.transform(test_df)
    feature_names_out = pre.get_feature_names_out()
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test_transformed)
    shap_results[fs_name] = (shap_values, feature_names_out, X_test_transformed)
    mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names_out).sort_values(ascending=False)
    print(f"=== {fs_name} / XGBoost - SHAP mean |value| ===")
    print(mean_abs.round(4).to_string())
    print()

=== A_Basic / XGBoost - SHAP mean |value| ===
num__Age                             0.9654
bin__H/O ChestPain                   0.7345
num__Weight (kg)                     0.5778
bin__Hypertension                    0.4894
num__BP(mmHg)                        0.4878
bin__Family H/O                      0.3850
bin__Diabetes                        0.3715
num__Height (cm)                     0.2196
cat__Sex_F                           0.0537
num__missingindicator_Height (cm)    0.0229
num__missingindicator_BP(mmHg)       0.0000
num__missingindicator_Weight (kg)    0.0000
cat__Sex_M                           0.0000



=== B_Basic_Enhanced / XGBoost - SHAP mean |value| ===
num__LDL(mg/dL)                                   1.4346
num__Triglycerides(mg/dL)                         0.9375
num__Total_Cholesterol(mg/dL)                     0.8150
num__RBS(mmol/L)                                  0.6662
num__Age                                          0.4273
num__BP(mmHg)                                     0.3492
bin__H/O ChestPain                                0.3486
bin__Hypertension                                 0.2679
num__Weight (kg)                                  0.1892
bin__Diabetes                                     0.1641
num__MaxHR                                        0.1376
num__Height (cm)                                  0.0676
num__HDL(mg/dL)                                   0.0603
bin__Family H/O                                   0.0432
num__missingindicator_Height (cm)                 0.0208
cat__Sex_F                                        0.0012
num__missingindicator_Triglycerid

SHAP rankings agree closely with the permutation-importance rankings
above for both models (Age/ChestPain/BP dominant in Basic; LDL/Triglycerides/
Total-Cholesterol/RBS dominant in Enhanced) - two independent importance
methods converging on the same answer is a useful robustness check.

## 5. Evidence-based trim: does removing MaxHR and HDL from Enhanced cost anything?

Per the instruction to let feature selection remove redundant/weak
features rather than assuming the original 6-field Enhanced list is final.

In [6]:
def make_pipeline(numeric_cols, algo):
    numeric_pipe = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler())])
    binary_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent'))])
    cat_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))])
    pre = ColumnTransformer([('num', numeric_pipe, numeric_cols), ('bin', binary_pipe, BASIC_BINARY), ('cat', cat_pipe, BASIC_CATEGORICAL)])
    spw = (train_df[TARGET]==0).sum() / (train_df[TARGET]==1).sum()
    model = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', scale_pos_weight=spw,
                           n_estimators=200, max_depth=3, learning_rate=0.05, n_jobs=-1)
    return Pipeline([('preprocess', pre), ('model', model)])

variants = {
    'B_full (incl. MaxHR, HDL)': BASIC_NUMERIC + ENHANCED_NUMERIC_ADD,
    'B_trimmed (LDL, Chol, Trig, RBS only)': BASIC_NUMERIC + ['Total_Cholesterol(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'RBS(mmol/L)'],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, cols in variants.items():
    pipe = make_pipeline(cols, 'xgb')
    cv_scores = cross_val_score(pipe, train_df, train_df[TARGET], cv=cv, scoring='roc_auc')
    pipe.fit(train_df, train_df[TARGET])
    proba = pipe.predict_proba(test_df)[:, 1]
    pred = pipe.predict(test_df)
    print(f"{name:40s} CV-AUC={cv_scores.mean():.4f}+/-{cv_scores.std():.4f}  Test-AUC={roc_auc_score(test_df[TARGET],proba):.4f}  Test-Recall={recall_score(test_df[TARGET],pred):.4f}")

B_full (incl. MaxHR, HDL)                CV-AUC=0.9817+/-0.0045  Test-AUC=0.9802  Test-Recall=0.9145


B_trimmed (LDL, Chol, Trig, RBS only)    CV-AUC=0.9815+/-0.0044  Test-AUC=0.9802  Test-Recall=0.9145


**Confirmed: dropping `MaxHR` and `HDL` costs nothing** - CV-AUC and
test-AUC/recall are effectively identical with and without them. **Final
recommended Enhanced/Optional feature list: Total Cholesterol, LDL,
Triglycerides, RBS** (4 fields, not 6).

## 6. Final recommended models: complete metrics on the held-out test set

In [7]:
def specificity(y_true, y_pred):
    tn = ((y_true==0)&(y_pred==0)).sum(); fp = ((y_true==0)&(y_pred==1)).sum()
    return tn/(tn+fp)

def full_eval(name, numeric_cols):
    pipe = make_pipeline(numeric_cols, 'xgb')
    pipe.fit(train_df, train_df[TARGET])
    proba = pipe.predict_proba(test_df)[:, 1]
    pred = pipe.predict(test_df)
    y = test_df[TARGET]
    print(f"=== {name} (XGBoost) - HELD-OUT TEST (n={len(test_df)}) ===")
    print(f"ROC-AUC={roc_auc_score(y,proba):.4f}  Accuracy={accuracy_score(y,pred):.4f}  "
          f"Precision={precision_score(y,pred):.4f}  Recall={recall_score(y,pred):.4f}  "
          f"Specificity={specificity(y,pred):.4f}  F1={f1_score(y,pred):.4f}  Brier={brier_score_loss(y,proba):.4f}")
    print("Confusion matrix [[TN FP][FN TP]]:\n", confusion_matrix(y, pred))
    print()

full_eval("FINAL Basic Model", BASIC_NUMERIC)
full_eval("FINAL Enhanced Model (trimmed)", BASIC_NUMERIC + ['Total_Cholesterol(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'RBS(mmol/L)'])

=== FINAL Basic Model (XGBoost) - HELD-OUT TEST (n=207) ===
ROC-AUC=0.9253  Accuracy=0.8454  Precision=0.8899  Recall=0.8291  Specificity=0.8667  F1=0.8584  Brier=0.1128
Confusion matrix [[TN FP][FN TP]]:
 [[78 12]
 [20 97]]



=== FINAL Enhanced Model (trimmed) (XGBoost) - HELD-OUT TEST (n=207) ===
ROC-AUC=0.9802  Accuracy=0.9324  Precision=0.9640  Recall=0.9145  Specificity=0.9556  F1=0.9386  Brier=0.0580
Confusion matrix [[TN FP][FN TP]]:
 [[ 86   4]
 [ 10 107]]



## 7. Summary

See `../reports/feature_selection.md` for the full written discussion, and
`../reports/ml_experiment_plan.md` for the complete methodology, all-model
comparison table, and final architecture recommendation.